# A chat app interface
A chat app with FastHTML and claudette/lisette

In [ ]:
#| default_exp chat

In [ ]:
#| hide
from IPython.display import HTML
from fasthtml.jupyter import *

In [ ]:
#| export
from fastlucide.icons import spritesheet, Eye_off, eye, pencil

In [ ]:
#| export
from fasthtml.common import *
from lisette import *
from fastlite import *
import random
import string
import json
from dataclasses import dataclass, field

## Creating the app with daisy styling -

In [ ]:
#| export
app = FastHTML(hdrs=(
        Link(href='https://cdn.jsdelivr.net/npm/daisyui@5', rel='stylesheet', type='text/css'),
        Link(href='https://cdn.jsdelivr.net/npm/daisyui@5/themes.css', rel='stylesheet', type='text/css'),
        Script(src='https://cdn.jsdelivr.net/npm/@tailwindcss/browser@4'),
        Link(rel='stylesheet', href='https://cdn.jsdelivr.net/npm/@tailwindcss/typography/dist/typography.min.css'),
        MarkdownJS(),
        ),
    )

In [ ]:
#| export
# For convenience
rt = app.route

In [ ]:
#| hide
#| eval: false
srv = JupyUvi(app)

In [ ]:
def preview(*c): return HTMX((*c,), app=app, host=None, port=None)

## Data types

In [ ]:
#| export
#| hide
class MsgType(enum.StrEnum):
    note = 'NOTE'
    prompt = 'PROMPT'
    ai = 'AI'

## App State and DB

In [ ]:
#| export
#| eval: false
db = database('solveish.db')

In [ ]:
#| export
class Dialog: id:int; uid:int; messages:str; name:str
class Message: id:int; dlgid:int; uid:int; content:str; mtype:str; hidden:bool

In [ ]:
#| export
#| eval: false
dialogs = db.create(Dialog, transform=True)
msgs = db.create(Message, transform=True)

In [ ]:
#| export
@dataclass
class AppState:
    uid: int = 123
    dlgid: int = field(default_factory=lambda: random.randint(1, 100000))
    dlgname: str = field(default_factory=lambda: ''.join(random.choices(string.ascii_lowercase, k=6)))
    model: str = 'claude-haiku-4-5'

    def randomize(self):
        self.dlgid = random.randint(1, 100000)
        self.dlgname = ''.join(random.choices(string.ascii_lowercase, k=6))
        return self

    def setdlg(self, dlgid):
        dlg = dialogs[dlgid]
        self.dlgid = dlg.id
        self.dlgname = dlg.name

In [ ]:
#| export
@dataclass
class MsgIn:
    msg:str; mtype:str; msgid:str
    
    def __post_init__(self):
        self.msgid = int(self.msgid) if self.msgid else None
        self.mtype = self.mtype.upper()

In [ ]:
#| export
def save_msg(d:MsgIn):
    if not state.dlgid in dialogs: dialogs.insert(id=state.dlgid, uid=state.uid, name=state.dlgname)
    dd = dict(dlgid=state.dlgid, uid=state.uid, content=d.msg, mtype=d.mtype)
    if d.msgid: return msgs.update(dd, d.msgid)
    return msgs.insert(dd)

In [ ]:
#| export
state = AppState()

## JS

In [ ]:
#| export
js_submit = '''
document.body.addEventListener('keydown', function(e) {
    if (e.target.matches('#msg')) {
        if ((e.shiftKey) && e.key === 'Enter') {
            e.preventDefault();
            document.getElementById('send').click()
        }
    }
});
'''

## Hiding messages

In [ ]:
#| export
def hideButton(msg):
    icon = Eye_off() if msg.hidden else eye()
    rt = unhide if msg.hidden else hide
    return Button(icon, cls='btn btn-ghost btn-xs', hx_post=rt, hx_vals=json.dumps(dict(msgid=msg.id)), hx_swap='outerHTML')

In [ ]:
#| export
@rt
def hide(msgid:int):
    m = msgs.update({'hidden': True}, msgid)
    return hideButton(m)

In [ ]:
#| export
@rt
def unhide(msgid:int):
    m = msgs.update({'hidden': False}, msgid)
    return hideButton(m)

## Editing messages

In [ ]:
#| export
@rt(methods='get')
def edit_msg(msgid:int):
    m = msgs[msgid]
    return main_form(m.mtype, m.content, m.id)

In [ ]:
#| export
def edit_btn(msg):
    icon = pencil()
    return Button(icon, cls='btn btn-ghost btn-xs', hx_get=edit_msg.to(msgid=msg.id), hx_target='#msg', hx_on__after_request="let ta = htmx.find('#msg'); ta.value = ta.textContent; ta.focus()")

## Msg Bubble

In [ ]:
#| export
def bubble(msg):
    cls = {MsgType.prompt:'bg-blue-100', MsgType.ai:'bg-base-300', MsgType.note:'bg-warning/20'}
    toolbar = Div(
        hideButton(msg),
        edit_btn(msg),
        cls='flex justify-end pt-1')
    return Div(
        Div(Div(msg.content, cls='marked prose'), cls=f'rounded-xl p-4 {cls[msg.mtype]}'
            ),
        toolbar,
        id=f'bubble-{msg.id}',
        cls='max-w-3xl mx-auto w-full'
    )

## LLM prompts

In [ ]:
#| export
def mk_prompt(msg, h):
    return "Past messages: " + json.dumps(h) + "\n\nCurrent question: " + msg

In [ ]:
mk_prompt("Hello", ["Old message"])

'Past messages: ["Old message"]\n\nCurrent question: Hello'

In [ ]:
#| export
def get_ctx(dlgid):
    h = msgs('dlgid=? AND hidden IS NOT 1', [dlgid], order_by='id')
    return [{'content': m.content, 'role': m.mtype} for m in h]

In [ ]:
#| export
sysp = "You are a chat model, in a chat with a user. Given the chat history as context, answer the user's current question concisely."

In [ ]:
#| export
@rt
def ask_llm(msg:str):
    ctx = get_ctx(state.dlgid)
    r = Chat(model=state.model, sp=sysp)(mk_prompt(msg, ctx))
    rsp = r.choices[0].message.content
    m = save_msg(MsgIn(rsp, MsgType.ai, None))
    return bubble(m)

In [ ]:
#| export
def llm_rsp(msg):
    return Div(
        Span(cls='loading loading-ring loading-md'), cls='flex justify-center',
        hx_post=ask_llm, hx_trigger='load', hx_swap='outerHTML',
        hx_vals=json.dumps(dict(msg=msg)))

## Main chat

In [ ]:
#| export
def next_form_type(mtype): return MsgType.prompt if mtype == MsgType.ai else mtype

In [ ]:
#| export
@rt(methods='post')
def upsert_message(d:MsgIn):
    is_edit = d.msgid is not None
    m = save_msg(d)
    res = [bubble(m), main_form(next_form_type(m.mtype))]
    if is_edit: res.append(HtmxResponseHeaders(retarget=f'#bubble-{m.id}', reswap='outerHTML'))
    elif m.mtype == MsgType.prompt: res.append(llm_rsp(m.content))
    return tuple(res)

In [ ]:
#| export
def main_form(mtype:MsgType=MsgType.prompt, txt_prefill:str='', msgid:int=None):
    if mtype == MsgType.ai:
        t = Input(type='hidden', name='mtype', value=MsgType.ai)
    else:
        t = (Label(Input(type='radio', name='mtype', value='prompt', checked='checked' if mtype == MsgType.prompt else None, cls='radio radio-primary'), 'Prompt', style="padding: 1rem;"),
                Label(Input(type='radio', name='mtype', value='note', checked='checked' if mtype == MsgType.note else None , cls='radio radio-secondary'), 'Note', style="padding: 1rem;"),)

    return Form(
            Div(
                t,
                Button('Submit', id='send', hx_post=upsert_message, hx_target='#chat', hx_swap='beforeend', cls='btn btn-primary', style="padding: 1rem;"),
                modeldropdown(),
                dlgdropdown(),
                style="padding: 1rem; display:flex; justify-content:center; align-items:center"
            ),
            Textarea(txt_prefill, id='msg', name='msg', type='text', placeholder='Type something...', style='width: 100%;', cls='input input-primary'),
            Input(type='hidden', id='msgid', name='msgid', value=msgid),
            hx_on__after_request="if(event.detail.successful) htmx.find('#msg').value=''",
            hx_swap_oob='true',
            id='main_form',
            style='position: fixed; bottom: 0; width: 100%; padding:1rem'
        )

In [ ]:
#| export
@rt
def load_dialog(dlgid:int):
    "Load dialog by id and return its message bubbles"
    state.setdlg(dlgid)
    h = msgs('dlgid=?', [state.dlgid], order_by='id')
    return (bubble(m) for m in h)


In [ ]:
#| export
def dlgdropdown():
    udlgs = dialogs('uid=?', [123])

    return Details(cls='dropdown dropdown-top')(
        Summary('Dialogues', cls='btn m-1'),
        Ul(cls='menu dropdown-content bg-base-100 rounded-box z-1 w-52 p-2 shadow-sm')(
            (Li(A(dlg.name, hx_post=load_dialog.to(dlgid=dlg.id), hx_target='#chat', hx_swap='innerHTML')) for dlg in udlgs)
        )
    )

In [ ]:
#| export
models = ['claude-haiku-4-5', 'claude-opus-4-5', 'claude-sonnet-4-5']
models

['claude-haiku-4-5', 'claude-opus-4-5', 'claude-sonnet-4-5']

In [ ]:
#| export
@rt
def setmodel(name:str):
    state.model = name
    return Summary(state.model, cls='btn m-1', id="selmodel")

In [ ]:
#| export
def modeldropdown():
    return Details(cls='dropdown dropdown-top')(
        Summary(state.model, cls='btn m-1', id="selmodel"),
        Ul(cls='menu dropdown-content bg-base-100 rounded-box z-1 w-52 p-2 shadow-sm')(
            (Li(A(m, hx_post=setmodel.to(name=m), hx_target='#selmodel', hx_swap="outerHTML", hx_on_click="this.closest('details').removeAttribute('open')")) for m in models)
        )
    )

In [ ]:
#| export
@rt
def chatpg():
    state.randomize()
    return Div(
        Div(id='chat', style='width:100%; box-sizing:border-box; padding:0 1rem; border: 1px solid #ccc; border-radius: 0.5rem; min-height:400px; height: calc(100vh - 158px); overflow-y: auto;')(
        ),
        main_form(),
        Script(js_submit),
        style='width: 100%'
    ), spritesheet

In [ ]:
#| hide
#| eval: false
srv.stop()

In [ ]:
#| hide
from nbdev import nbdev_export; nbdev_export()